# 1. Load Data from all the stock csv files in directory

In [1]:
import datetime
import glob
import numpy as np
import pandas as pd


In [3]:

# Get CSV files list from a folder
csv_files = glob.glob("../data/raw/TSLA/TSLA-BID_ASK-*.csv")

# Read each CSV file into DataFrame
# This creates a list of dataframes
df_list = (pd.read_csv(file) for file in csv_files)

# Concatenate all DataFrames
df   = pd.concat(df_list, ignore_index=True)

## 1.1 Drop duplicates & check for missing values

In [ ]:
dups = len(df['date'])-len(df['date'].drop_duplicates())
print("Before: # of duplicates", dups, ' out of ', len(df) , ' or ', round(dups/len(df),5), '%')

In [ ]:
# Drop duplicate entries
df.drop_duplicates(subset=['date'], keep='first', inplace=True)

In [ ]:
# verify there are no duplicate values
df["date"].is_unique
dups = len(df['date'])-len(df['date'].drop_duplicates())
print("After: # of duplicates", dups, ' out of ', len(df) , ' or ', round(dups/len(df),5), '%')

## 1.2 Print out all dates with confirming # of quotes (23400) and non-conforming number of quotes

In [ ]:
df['date2_str']= df['date'][::].str.slice(stop=10)
pd_group_cnt = df.groupby(['date2_str'])['date2_str'].count().to_frame()
print("Confirming / correct number of quotes: 23,400")
print(pd_group_cnt.loc[pd_group_cnt['date2_str']==23400] )
pd.set_option('display.max_rows', None)
print("Dates Missing quotes")
print(pd_group_cnt.loc[pd_group_cnt['date2_str']!=23400] )
pd.set_option('display.max_rows', 10)
df = df.drop('date2_str', axis=1)

In [ ]:
# Only want to track average to 3 decimal places.  Otherwise, end up with a lot of digits
df['average'] = df['average'].round(decimals = 3)

# 2.0 Describe data

In [ ]:
df = df.sort_index(ascending=True)
#df = df.tail(10000)
#df = df.tail(30000)
df.describe()

In [ ]:
df.info()

# 3.0 Add computed columns

In [ ]:
df.set_index('date')
df = df.sort_index()

In [ ]:
# lambda functions

#return 1st value in series
def firstValue(rows):
    return rows.iloc[0]

#return last value in series
def lastValue(rows):
    return rows.iloc[-1]

#return arrow indicator for boxed in values;
#   -1 below lower bound
#    0 inside the box
#   +1 above the max value
def arrow(new_amt, old_amt, box):
    if old_amt == np.nan:
        return np.nan
    if new_amt == np.nan:
        return np.nan
    if (new_amt - old_amt) <= (box * -1):
        return '-1'
    if (new_amt - old_amt) >= box:
        return '1'
    else:
        return '0'

#  lambda function to adds up the last 5 values, excluding the very last value
def sum_last_5(rows):
    #print ("[" , rows[-6:-1], rows[-6:-1].sum(), "]")
    return rows[-6:-1].sum()

#  lambda function returns lowest of the last 5 values, excluding the very last value
def min_last_5(rows):
    return rows.iloc[-6:-1].min()

#  lambda function returns higest of  the last 5 values, excluding the very last value
def max_last_5(rows):
    return rows.iloc[-6:-1].max()

# Store/Save 1 second windows for h1, h2, h3, h4, h5

In [ ]:
%%time
df['h1s_high_max'] = df['high'].rolling(window=2).agg( {'maxLast': firstValue})
df['h1s_low_min'] = df['low'].rolling(window=2).agg( {'minLast': firstValue})
df['h1s_barCount_sum'] = df['barCount'].rolling(window=2).agg( {'sumLast': firstValue})
df['h1s_volume_sum'] = df['volume'].rolling(window=2).agg( {'sumLast': firstValue})
df['h1s_average_avg'] = df['average'].rolling(window=2).agg( {'sumLast': firstValue})

In [ ]:
%%time
df['h2s_high_max'] = df['high'].rolling(window=3).agg( {'maxLast': firstValue})
df['h2s_low_min'] = df['low'].rolling(window=3).agg( {'minLast': firstValue})
df['h2s_barCount_sum'] = df['barCount'].rolling(window=3).agg( {'sumLast': firstValue})
df['h2s_volume_sum'] = df['volume'].rolling(window=3).agg( {'sumLast': firstValue})
df['h2s_average_avg'] = df['average'].rolling(window=3).agg( {'sumLast': firstValue})


In [ ]:
%%time
df['h3s_high_max'] = df['high'].rolling(window=4).agg( {'maxLast': firstValue})
df['h3s_low_min'] = df['low'].rolling(window=4).agg( {'minLast': firstValue})
df['h3s_barCount_sum'] = df['barCount'].rolling(window=4).agg( {'sumLast': firstValue})
df['h3s_volume_sum'] = df['volume'].rolling(window=4).agg( {'sumLast': firstValue})
df['h3s_average_avg'] = df['average'].rolling(window=4).agg( {'sumLast': firstValue})


In [ ]:
%%time
df['h4s_high_max'] = df['high'].rolling(window=5).agg( {'maxLast': firstValue})
df['h4s_low_min'] = df['low'].rolling(window=5).agg( {'minLast': firstValue})
df['h4s_barCount_sum'] = df['barCount'].rolling(window=5).agg( {'sumLast': firstValue})
df['h4s_volume_sum'] = df['volume'].rolling(window=5).agg( {'sumLast': firstValue})
df['h4s_average_avg'] = df['average'].rolling(window=5).agg( {'sumLast': firstValue})

In [ ]:
df = df.drop('date2_str', axis=1)

## Compute 5 second window summary

In [ ]:
%%time
df['h5s_high_max'] = df['high'].rolling(window=6).agg( {'maxLast5': max_last_5})
df['h5s_low_min'] = df['low'].rolling(window=6).agg( {'minLast5': min_last_5})
df['h5s_barCount_sum'] = df['barCount'].rolling(window=6).agg( {'sumLast5': sum_last_5})
df['h5s_volume_sum'] = df['volume'].rolling(window=6).agg( {'sumLast5': sum_last_5})
df['_h5s_weighted_vol_avg_sum'] = df['_weighted_vol_avg'].rolling(window=6).agg({'SumLast5': sum_last_5})

df['h5s_average_avg'] = df['_h5s_weighted_vol_avg_sum'] / df['h5s_volume_sum']
df['h5s_average_avg'] = df['h5s_average_avg'].round(decimals = 3)
df.drop(columns=['_h5s_weighted_vol_avg_sum'])
#

## Compute 10 second window summary

In [ ]:
%%time
df['h10s_high_max'] = df['high'].rolling(window=11).agg( {'maxLast5': max_last_5})
df['h10s_low_min'] = df['low'].rolling(window=11).agg( {'minLast5': min_last_5})
df['h10s_barCount_sum'] = df['barCount'].rolling(window=11).agg( {'sumLast5': sum_last_5})
df['h10s_volume_sum'] = df['volume'].rolling(window=11).agg( {'sumLast5': sum_last_5})
df['_h10s_weighted_vol_avg_sum'] = df['_weighted_vol_avg'].rolling(window=11).agg({'SumLast5': sum_last_5})

df['h10s_average_avg'] = df['_h10s_weighted_vol_avg_sum'] / df['h10s_volume_sum']
df['h10s_average_avg'] = df['h10s_average_avg'].round(decimals = 3)
df.drop(columns=['_h10s_weighted_vol_avg_sum'])

## Compute 15 second window summary

In [ ]:
%%time
df['h15s_high_max'] = df['high'].rolling(window=16).agg( {'maxLast5': max_last_5})
df['h15s_low_min'] = df['low'].rolling(window=16).agg( {'minLast5': min_last_5})
df['h15s_barCount_sum'] = df['barCount'].rolling(window=16).agg( {'sumLast5': sum_last_5})
df['h15s_volume_sum'] = df['volume'].rolling(window=16).agg( {'sumLast5': sum_last_5})
df['_h15s_weighted_vol_avg_sum'] = df['_weighted_vol_avg'].rolling(window=16).agg({'SumLast5': sum_last_5})

df['h15s_average_avg'] = df['_h15s_weighted_vol_avg_sum'] / df['h15s_volume_sum']
df['h15s_average_avg'] = df['h15s_average_avg'].round(decimals = 3)
df.drop(columns=['_h15s_weighted_vol_avg_sum'])

In [ ]:
%%time
df = df.drop(columns=['_weighted_vol_avg'])

In [ ]:
df.set_index('date')
df = df.sort_index(ascending=False)
df.reset_index()
df.head(100)

## Compute Future 5 second window summary

In [ ]:
%%time
df['f5s_average'] = df['average'].rolling(window=6).agg( {'firstValue': firstValue})
df.head(100)

In [ ]:
df = df.sort_index(ascending=True)
df.reset_index()
df.head(100)

In [ ]:
%%time
pd.set_option('display.max_rows', 100)

df['f5s_10c_arrow'] = df.apply(lambda x: arrow(x['f5s_average'], x['average'], 0.10), axis=1)
df['f5s_15c_arrow'] = df.apply(lambda x: arrow(x['f5s_average'], x['average'], 0.15), axis=1)
df['f5s_20c_arrow'] = df.apply(lambda x: arrow(x['f5s_average'], x['average'], 0.20), axis=1)
df['f5s_25c_arrow'] = df.apply(lambda x: arrow(x['f5s_average'], x['average'], 0.25), axis=1)
df['f5s_30c_arrow'] = df.apply(lambda x: arrow(x['f5s_average'], x['average'], 0.30), axis=1)


In [ ]:
df.head(100)

In [ ]:
df.to_csv("./TSLA-BID_ASK.csv", index=False)